# Tester la branche regex

Ce notebook ne lit pas de PDF. Il teste uniquement la branche `regex` du pipeline sur les cas centralises dans `configs/test_cases.csv`.

La branche detecte les emails, telephones francais, IBAN francais et NIR francais avec des expressions regulieres locales. Aucun modele n'est telecharge ni charge.


In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SRC_DIR = ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

pd.set_option("display.max_colwidth", 180)

BRANCH_NAME = "regex"
ENABLED_BRANCHES = (BRANCH_NAME,)
BRANCH_SCORE_COLUMN = "regex_score"
MODEL_CACHE_DIR = r"D:\Workspaces\ModelCache"
MODEL_STORE_DIR = r"D:\Workspaces\modelStore"

os.environ["HF_HOME"] = MODEL_CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = rf"{MODEL_CACHE_DIR}\transformers"
os.environ["HUGGINGFACE_HUB_CACHE"] = rf"{MODEL_CACHE_DIR}\hub"
os.environ["COMPLIANCE_NLP_MODEL_CACHE"] = MODEL_CACHE_DIR
os.environ["COMPLIANCE_NLP_MODEL_STORE"] = MODEL_STORE_DIR

params = {
    "enabled_branches": ENABLED_BRANCHES,
    "branch_score_column": BRANCH_SCORE_COLUMN,
    "model_cache_dir": MODEL_CACHE_DIR,
    "model_store_dir": MODEL_STORE_DIR,
    "downloads": "none",
}
params


In [ ]:
from compliance_nlp import (
    load_generic_detection_rules,
    load_section_definitions,
    load_whitelist_terms,
)
from compliance_nlp.pipeline import analyze_text

sections = load_section_definitions(ROOT / "configs" / "sections.csv")
generic_rules = load_generic_detection_rules(ROOT / "configs" / "generic_detection_rules.csv")
whitelist_terms = load_whitelist_terms(ROOT / "configs" / "article9_whitelist.csv")

pd.DataFrame([
    {"referentiel": "sections", "count": len(sections)},
    {"referentiel": "generic_rules", "count": len(generic_rules)},
    {"referentiel": "whitelist_terms", "count": len(whitelist_terms)},
])


In [ ]:
TEST_CASES_PATH = ROOT / "configs" / "test_cases.csv"
TEST_CASE_COLUMNS = ["text", "control_family", "comment"]

test_cases = pd.read_csv(TEST_CASES_PATH, keep_default_na=False)
missing_columns = set(TEST_CASE_COLUMNS) - set(test_cases.columns)
if missing_columns:
    raise ValueError(f"Colonnes manquantes dans {TEST_CASES_PATH}: {sorted(missing_columns)}")

test_cases = test_cases[TEST_CASE_COLUMNS].copy()
test_cases["case_id"] = [f"case_{index + 1:02d}" for index in range(len(test_cases))]
test_cases = test_cases[["case_id", "text", "control_family", "comment"]]
test_cases


In [ ]:
def finding_key(finding):
    return finding.rule_id or finding.code


def run_detector(text, case_id):
    return analyze_text(
        document_name=case_id,
        source_path="notebook",
        extracted_text=text,
        generic_rules=generic_rules,
        section_definitions=sections,
        whitelist_terms=whitelist_terms,
        enabled_branches=ENABLED_BRANCHES,
    )


def findings_to_rows(case, analysis):
    findings = analysis.findings
    if not findings:
        return [{
            "case_id": case["case_id"],
            "control_family": case["control_family"],
            "comment": case["comment"],
            "detection_engine": BRANCH_NAME,
            "predicted_key": None,
            "code": None,
            "rule_id": None,
            "section": None,
            "category": None,
            "matched_term": None,
            "detection_type": None,
            "score": None,
            "branch_score": None,
            "regex_score": None,
            "severity": None,
            "evidence": None,
        }]

    return [{
        "case_id": case["case_id"],
        "control_family": case["control_family"],
        "comment": case["comment"],
        "detection_engine": finding.detection_engine,
        "predicted_key": finding_key(finding),
        "code": finding.code,
        "rule_id": finding.rule_id,
        "section": finding.section,
        "category": finding.category,
        "matched_term": finding.matched_term,
        "detection_type": finding.detection_type,
        "score": finding.score,
        "branch_score": finding.branch_score,
        "regex_score": finding.regex_score,
        "severity": finding.severity,
        "evidence": finding.evidence,
    } for finding in findings]


In [ ]:
rows = []
metadata_rows = []
for _, case in test_cases.iterrows():
    analysis = run_detector(case["text"], case["case_id"])
    rows.extend(findings_to_rows(case, analysis))
    metadata_rows.append({
        "case_id": case["case_id"],
        "control_family": case["control_family"],
        "comment": case["comment"],
        "regex_finding_count": analysis.metadata.get("regex_finding_count"),
        "regex_max_score": analysis.metadata.get("regex_max_score"),
        "branch_errors": analysis.metadata.get("branch_errors"),
    })

results_df = pd.DataFrame(rows)
summary_df = pd.DataFrame(metadata_rows)
summary_df


In [ ]:
alerts_df = results_df[results_df["predicted_key"].notna()].copy()
alerts_df.sort_values(["case_id", "score"], ascending=[True, False])


In [ ]:
entity_summary_df = (
    alerts_df.groupby(["predicted_key"], dropna=False)
    .agg(
        detection_count=("predicted_key", "size"),
        max_score=("regex_score", "max"),
        matched_terms=("matched_term", lambda values: " | ".join(sorted({value for value in values if value}))),
    )
    .reset_index()
    .sort_values(["predicted_key"])
)
entity_summary_df
